Optional homework: random vs chronological vs customer-grouped split.

# Invoice payment timing — simplified classroom pipeline

**Instrumentarium:** Lessmann, S., Baesens, B., Seow, H.-V., & Thomas, L. C. (2015). *Benchmarking state-of-the-art classification algorithms for credit scoring: An update of research.* *EJOR*, 247(1), 124–136. Run the protocol in `03b`.

**Case:** B2B invoice timing (“when will you pay?”).  
**Ancestry only:** Zeng, Melville et al. (2008) *KDD* posed this task — not the current methods paper.

Pedagogical sketch with **synthetic invoices**. Practice:

1. invoice-level labels (binary `late` + delay buckets);
2. customer-history features without obvious leakage;
3. **time-based** train/val/test;
4. logistic → RF → GBT;
5. evaluation via **AUC / AP** and an explicit **cost matrix**.

Classroom DGP: **most invoices on time**, late is the minority (realistic imbalance).

**Companion:** `sessions/03_credit_risk.md`  
**Next:** Lessmann protocol — `notebooks/03b_lessmann_protocol.ipynb`

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score, confusion_matrix
)

rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

## 1. Synthetic B2B invoice panel

- \(C\) customers with latent payment quality.
- Features use **past** payment behaviour only (no future leakage).
- Outcomes: `days_late`, binary `late`, ordered `delay_bucket`.

In [ ]:

n_customers = 150
n_invoices = 5000

cust_quality = rng.normal(size=n_customers)          # higher => pays better
cust_industry = rng.integers(0, 4, size=n_customers)

invoice_day = rng.integers(0, 900, size=n_invoices)
customer_id = rng.integers(0, n_customers, size=n_invoices)
amount = rng.lognormal(mean=8.0, sigma=0.7, size=n_invoices)
terms_days = rng.choice([30, 45, 60], size=n_invoices, p=[0.55, 0.30, 0.15])

order = np.argsort(invoice_day)
invoice_day = invoice_day[order]
customer_id = customer_id[order]
amount = amount[order]
terms_days = terms_days[order]

days_late = np.empty(n_invoices)
hist_late_rate = np.zeros(n_invoices)
hist_avg_late = np.zeros(n_invoices)
hist_n_prior = np.zeros(n_invoices)
hist_open_amt = np.zeros(n_invoices)

sum_late = np.zeros(n_customers)
sum_days = np.zeros(n_customers)
n_prior = np.zeros(n_customers)
open_amt = np.zeros(n_customers)

for i in range(n_invoices):
    c = customer_id[i]
    hist_late_rate[i] = sum_late[c] / n_prior[c] if n_prior[c] > 0 else 0.0
    hist_avg_late[i] = sum_days[c] / n_prior[c] if n_prior[c] > 0 else 0.0
    hist_n_prior[i] = n_prior[c]
    hist_open_amt[i] = open_amt[c]

    # latent delinquency score → mostly on-time, minority late
    score = (
        -1.2 * cust_quality[c]
        + 0.35 * np.log(amount[i] / np.median(amount))
        + 0.015 * (terms_days[i] - 30)
        + 0.9 * hist_late_rate[i]
        + 0.015 * hist_avg_late[i]
        + 0.10 * (cust_industry[c] == 3)
        + rng.normal(scale=1.05)
    )
    # days late: most invoices on time; late is minority
    if score < 1.1:
        dl = 0.0
    else:
        dl = float(np.clip(rng.lognormal(mean=2.0 + 0.30 * (score - 1.1), sigma=0.60), 1, 120))

    days_late[i] = dl
    is_late = float(dl > 0)
    sum_late[c] += is_late
    sum_days[c] += dl
    n_prior[c] += 1.0
    # open exposure proxy: decays + new amount if late-prone
    open_amt[c] = 0.85 * open_amt[c] + (amount[i] if dl > 15 else 0.15 * amount[i])

def bucket(dl):
    if dl <= 0:
        return "on_time"
    if dl <= 30:
        return "d1_30"
    if dl <= 60:
        return "d31_60"
    return "d60p"

inv = pd.DataFrame({
    "invoice_day": invoice_day,
    "customer_id": customer_id,
    "amount": amount,
    "terms_days": terms_days,
    "industry": cust_industry[customer_id],
    "hist_late_rate": hist_late_rate,
    "hist_avg_late": hist_avg_late,
    "hist_n_prior": hist_n_prior,
    "hist_open_amt": hist_open_amt,
    "days_late": days_late,
})
inv["late"] = (inv["days_late"] > 0).astype(int)
inv["delay_bucket"] = inv["days_late"].map(bucket)
# drop cold-start invoices with no history (optional clarity)
inv = inv[inv["hist_n_prior"] >= 1].reset_index(drop=True)

features = [
    "amount", "terms_days", "industry",
    "hist_late_rate", "hist_avg_late", "hist_n_prior", "hist_open_amt",
]

inv.head()

In [ ]:
print(inv.shape)
print("P(late) =", round(inv["late"].mean(), 3))
print(inv["delay_bucket"].value_counts(normalize=True).round(3))
inv["days_late"].describe().round(2)

## 2. Time-based split (by `invoice_day`)

Do **not** randomly shuffle invoices — that leaks customer futures into the past.

In [ ]:
train = inv[inv["invoice_day"] < 550]
val = inv[(inv["invoice_day"] >= 550) & (inv["invoice_day"] < 720)]
test = inv[inv["invoice_day"] >= 720]

for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s}: n={len(df):5d} | customers={df['customer_id'].nunique():3d} | P(late)={df['late'].mean():.3f}")

X_train, y_train = train[features], train["late"].to_numpy()
X_val, y_val = val[features], val["late"].to_numpy()
X_test, y_test = test[features], test["late"].to_numpy()

## 3. Model ladder (binary late)

In [ ]:
candidates = {
    "Logit": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=500, class_weight="balanced")),
    ]),
    "RF": RandomForestClassifier(
        n_estimators=250, max_depth=6, min_samples_leaf=30,
        class_weight="balanced", random_state=42, n_jobs=-1,
    ),
    "GBT": GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05, random_state=42,
    ),
}

fitted = {}
rows = []
for name, model in candidates.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    proba = model.predict_proba(X_val)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows.append({
        "model": name,
        "acc_val": accuracy_score(y_val, pred),
        "auc_val": roc_auc_score(y_val, proba),
        "ap_val": average_precision_score(y_val, proba),
    })
    print(f"{name:5s}  AUC={rows[-1]['auc_val']:.3f}  AP={rows[-1]['ap_val']:.3f}")

val_board = pd.DataFrame(rows).sort_values("auc_val", ascending=False)
val_board.round(3)

## 4. Economic evaluation: expected cost

Action: contact / prioritize if \(\hat p \ge \tau\).

|  | Predict on-time (no action) | Predict late (action) |
|--|-----------------------------|------------------------|
| Actually on-time | 0 | \(c_{FP}=1\) |
| Actually late | \(c_{FN}=5\) | \(c_{TP}=0.5\) (action cost) |

Tune \(\tau\) on validation by minimizing expected cost; report on test.

In [ ]:
def expected_cost(y_true, proba, tau=0.5, c_fp=1.0, c_fn=5.0, c_tp=0.5):
    y_hat = (proba >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
    total = c_fp * fp + c_fn * fn + c_tp * tp
    return total / len(y_true), dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))


cost_rows = []
for name, model in fitted.items():
    p_val = model.predict_proba(X_val)[:, 1]
    p_test = model.predict_proba(X_test)[:, 1]
    best_tau, best_c = 0.5, np.inf
    for tau in np.linspace(0.2, 0.8, 13):
        c, _ = expected_cost(y_val, p_val, tau=tau)
        if c < best_c:
            best_c, best_tau = c, tau
    c_test, cm = expected_cost(y_test, p_test, tau=best_tau)
    cost_rows.append({
        "model": name,
        "tau_val": best_tau,
        "auc_test": roc_auc_score(y_test, p_test),
        "ap_test": average_precision_score(y_test, p_test),
        "exp_cost_test": c_test,
        **cm,
    })

cost_board = pd.DataFrame(cost_rows).sort_values("exp_cost_test")
cost_board.round(3)

In [ ]:
best = cost_board.iloc[0]["model"]
p = fitted[best].predict_proba(X_test)[:, 1]
taus = np.linspace(0.1, 0.9, 17)
curve = [expected_cost(y_test, p, tau=t)[0] for t in taus]
plt.plot(taus, curve, marker="o")
plt.xlabel("threshold τ")
plt.ylabel("expected cost (test)")
plt.title(f"Cost–threshold curve — {best}")
plt.tight_layout()
plt.show()

## 5. Discussion questions

1. Which leakage appears if we use random `train_test_split` on invoices?
2. Why might **AP / PR** matter more than AUC under imbalance?
3. If GBT wins AUC but Logit wins expected cost, what do you deploy?
4. How would **delay buckets** change collector playbooks vs binary `late`?
5. Map to credit default: what is \(c_{FN}\)?

## 6. Experiments — "do differently"

In [ ]:
# A) multiclass delay_bucket
# B) customer-level blocked split
# C) harsher FN cost
# D) drop history features

c_fn_alt = 10.0
name = "Logit"
p_val = fitted[name].predict_proba(X_val)[:, 1]
p_test = fitted[name].predict_proba(X_test)[:, 1]
best_tau, best_c = 0.5, np.inf
for tau in np.linspace(0.2, 0.8, 13):
    c, _ = expected_cost(y_val, p_val, tau=tau, c_fn=c_fn_alt)
    if c < best_c:
        best_c, best_tau = c, tau
c_test, cm = expected_cost(y_test, p_test, tau=best_tau, c_fn=c_fn_alt)
print(f"{name}: c_fn={c_fn_alt}, tau={best_tau:.2f}, exp_cost={c_test:.3f}, cm={cm}")

## Takeaway

Invoice timing is a **decision problem**: classifiers matter, but the operating threshold and cost matrix decide value.

**Next:** Lessmann — compare models under a fixed benchmarking protocol.